# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amin119/FlyRank-AI-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring** — built on top of a **binary classification** sub-task.

The decision from Week 1 is "which pages does a capacity-limited reviewer look at first, this week." That's not a yes/no question asked of one page in isolation — it's an ordering problem over the whole inventory, consumed top-down until capacity runs out. That matches the framing skill's task-type table exactly: the question sounds like "which ones first?", the target is a priority score, and the metric is precision@K.

It's worth being precise here, because it's easy to blur two different questions. The *mechanism* that produces the score is a binary classifier — a model trained to output P(this page is declining), exactly like the starter pipeline's logistic regression / decision tree / random forest already do. That output is a probability, not a queue position. Turning it into a queue is a separate step: rank all 30,000 probabilities high-to-low, optionally blend with the transparent baseline score (the starter's `final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)`), and hand a reviewer the top K.

So: the ML **task type** the business needs is ranking/scoring. The **model type** I'll train to get there is a binary classifier. Conflating the two is a common beginner mistake — "I'll train a classifier" answers "how," not "what for." I'm answering "what for" here: an ordered, capacity-aware review queue.

It is explicitly **not**:
- Clustering — I'm not describing archetypes of pages (that's Lane 3); I'm prioritizing specific pages for a specific action.
- Plain signal analysis — I'm not just reporting "X correlates with Y" (that's Lane 1); the output has to be something a reviewer acts on, in order.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter proxy target:** `is_declining_label = (trend_direction == "down")`, i.e. `impressions_last_30d` fell more than 20% versus `impressions_prev_30d`, both measured inside the same trailing-90-day export window.

I rebuilt this by hand from the two raw impression columns in the code cell directly below, and it matches the shipped `trend_direction` on 100% of the 30,000 rows (0 mismatches) — so I'm confident I understand exactly what this label is, not just what it's called.

And what it is, is a **defined-rule proxy, not an observed future outcome.** `trend_pct` and `trend_direction` are both computed *inside* the same 90-day snapshot the features come from — "last 30 days" isn't a future checkpoint relative to some earlier decision point, it's just the tail end of the same export. That has two consequences:
- Per the data skill's "label trap": `trend_pct` and `trend_direction` can never be model features. They don't just correlate with the label — they *define* it. Using either as a feature would let the model read the answer straight off an input column (a circular result, not a discovery). The code cell below checks this explicitly.
- Per the lane guide, this is explicitly flagged as "a beginner proxy label" — useful for learning the pipeline end to end, but weaker than what a real capstone should stand on.

**Where I'm heading once I have warehouse access (ML-04+):** a genuinely forward-looking label, built from `fact_content_daily_performance` — features computed from a prior window (e.g. days −119 to −30 relative to a chosen cutoff date), predicting an outcome measured in a strictly *later*, non-overlapping window (the next 30 days after that cutoff). That label would be observed *after* the decision point, which is the only kind the "target must be observed, not defined" rule actually trusts. Until I build that, I'm using the starter's current-window proxy with my eyes open about its limits — not treating it as the finish line.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/amin119/FlyRank-AI-internship"
REPO_DIR = "FlyRank-AI-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks" and os.path.basename(os.path.dirname(os.getcwd())) == "work":
    os.chdir("../..")  # move from work/notebooks/ to the repo root

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ---- Rebuild the proxy target BY HAND, from its raw inputs, to prove what it actually is ----
recomputed_pct = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"].replace(0, np.nan) * 100
)
recomputed_declining = (recomputed_pct < -20).fillna(False)

is_declining_label = (df["trend_direction"] == "down").astype(int)

agree_rate = (recomputed_declining.astype(int) == is_declining_label).mean()
mismatches = (recomputed_declining.astype(int) != is_declining_label).sum()
print(f"Hand-rebuilt label agrees with the shipped trend_direction on {agree_rate * 100:.1f}% of rows "
      f"({mismatches} mismatches out of {len(df):,}).")

print("\nTarget sketch -- is_declining_label value counts:")
print(is_declining_label.value_counts().rename({1: "declining (1)", 0: "not declining (0)"}))
print(f"\nBase rate: {is_declining_label.mean() * 100:.1f}% of pages currently carry the proxy label.")

# ---- Prove the label-source columns are excluded from any feature list ----
label_source_cols = {"trend_pct", "trend_direction"}
candidate_feature_cols = {
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "days_since_last_update", "word_count", "content_age_days",
}
print("\nLabel-source columns kept OUT of features:", label_source_cols)
print("Leakage check -- any overlap with candidate features?",
      "FAIL" if label_source_cols & candidate_feature_cols else "none (clean)")

Hand-rebuilt label agrees with the shipped trend_direction on 100.0% of rows (0 mismatches out of 30,000).

Target sketch -- is_declining_label value counts:
trend_direction
declining (1)        16262
not declining (0)    13738
Name: count, dtype: int64

Base rate: 54.2% of pages currently carry the proxy label.

Label-source columns kept OUT of features: {'trend_direction', 'trend_pct'}
Leakage check -- any overlap with candidate features? none (clean)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

The decision this serves is "a reviewer works through the top of a ranked queue until their capacity runs out." Fifty is the same capacity the starter pipeline's own committed report already evaluates against (`outputs/model_report.md`), so my number is directly comparable to a real, already-computed baseline: **baseline rule precision@50 = 0.240** — of the top 50 pages the fixed rule ranks first, only about 12 actually carry the label. That's the bar any model I build has to clear on an honest split (client holdout) to be worth using at all.

Why this metric and not the obvious alternatives:
- **Not plain accuracy or ROC-AUC** — both score the *whole* ranking evenly, but the reviewer never sees the whole ranking. They see the top 50. A model can be mediocre everywhere else and still be exactly what's needed if its top 50 are excellent.
- **Not recall alone** — from Week 1's cost analysis, missing a real decline is costly, but so is burning reviewer trust on false alarms; recall on its own ignores the second cost entirely.

I'll still *track* Average Precision (in case the reviewer's real capacity turns out to be 20 or 100, not 50) and Recall (as a check that I'm not just getting lucky at K=50) — but Precision@50 is the one number I'd defend if someone asked "is this good," because it's the one that matches exactly what gets used.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
from IPython.display import display

# df and is_declining_label already exist from Section 2's code cell -- reused here, not reloaded.

# ---- The unit of analysis: one row = one page (one pseudonymized content item) ----
lane_cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "days_since_last_update", "word_count", "content_age_days",
    "impressions_last_30d", "impressions_prev_30d", "trend_pct", "trend_direction",
]
lane_slice = df[lane_cols].copy()
print(f"Lane 2 slice: {lane_slice.shape[0]:,} rows x {lane_slice.shape[1]} columns")
print("One row = one page. Grain check (content_id unique):", lane_slice["content_id"].is_unique)
display(lane_slice.head(5))

# ---- Why no single signal is a safe substitute for a model: correlation with the label ----
numeric_signals = ["impressions_90d", "clicks_90d", "avg_position", "ctr",
                    "days_since_last_update", "word_count", "content_age_days"]
corr_rows = []
for col in numeric_signals:
    mask = df[col].notna()
    corr = np.corrcoef(df.loc[mask, col], is_declining_label[mask])[0, 1]
    corr_rows.append({"signal": col, "corr_with_label": round(corr, 3)})

corr_table = pd.DataFrame(corr_rows).sort_values("corr_with_label", key=abs, ascending=False)
print("\nEach signal's correlation with the proxy label, taken alone:")
display(corr_table)
print(f"Strongest single signal: {corr_table.iloc[0]['signal']} at r = {corr_table.iloc[0]['corr_with_label']}"
      " -- weak, by any standard.")

Lane 2 slice: 30,000 rows x 13 columns
One row = one page. Grain check (content_id unique): True


,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,word_count,content_age_days,impressions_last_30d,impressions_prev_30d,trend_pct,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,20,3221.0,187,578,987,-41.4,down
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,25,2481.0,445,2501,5915,-57.7,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,20,3515.0,141,2382,6089,-60.9,down
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,22,NaN,463,3626,4206,-13.8,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,14,2803.0,263,4211,6452,-34.7,down



Each signal's correlation with the proxy label, taken alone:


,signal,corr_with_label
6,content_age_days,-0.164
5,word_count,0.090
4,days_since_last_update,0.081
3,ctr,-0.062
1,clicks_90d,-0.040
2,avg_position,-0.029
0,impressions_90d,-0.018


Strongest single signal: content_age_days at r = -0.164 -- weak, by any standard.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Because the individual signals are each weak, but the combination isn't.**

Section 4's correlation table makes this concrete: taken one at a time, against the proxy label, every obvious numeric signal is weak — the strongest single one (`content_age_days`, r ≈ −0.16) is nowhere near strong enough to threshold on by itself, and most of the others (`impressions_90d`, `avg_position`, `ctr`, `clicks_90d`) sit under |r| = 0.06. If I tried to write "if signal X > threshold, flag for review" for any one of these alone, I'd be building a rule on noise.

Yet Week 1's notebook already showed that *combining* these same weak, correlated signals — with a model instead of one threshold — jumps Precision@50 from 0.240 (the fixed rule) to 0.740 (a random forest): roughly 3x more true positives in the exact list length a reviewer works through, on the exact same underlying data. No new information was added; the only thing that changed was *how* the existing weak signals were combined and weighted.

That's the actual argument for ML over a fixed rule here, and it's a narrower claim than "ML is smarter": the pattern is real (the jump is measured, not assumed), but it's built from many partial, interacting, non-linear pieces of evidence — staleness only matters if demand is still there; a CTR gap only means something relative to its position tier; word count's relevance depends on content type. A hand-written if-statement can encode one or two of those interactions. Weighing seven-plus of them correctly, differently for every page, across 30,000 pages, is exactly the "many signals, tangled, shifting over time" case the framing skill names as where a model earns its place — and here, unusually for Week 2, I already have measured evidence that it does, on this slice, under a fair split.

What I still can't claim: that this holds on the full warehouse, that the *proxy* label from Section 2 is the right target long-term, or that any of this is causal. Those are exactly the open questions the data contract, signal audit, and warehouse-label work in the coming weeks exist to close.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.